# FleetFuel360 - Exploratory Data Analysis

This notebook provides comprehensive analysis of fuel efficiency data for logistics fleet management.

## Objectives:
1. Load and explore fuel logs data
2. Analyze fuel efficiency patterns
3. Identify correlations and trends
4. Vehicle performance comparison
5. Anomaly detection visualization
6. Generate insights for fleet optimization

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("📊 FleetFuel360 - Exploratory Data Analysis")
print("=" * 50)

## 1. Data Loading and Initial Exploration

In [ ]:
# Load the fuel logs data
data_path = '../data/fuel_logs_sample.csv'
df = pd.read_csv(data_path)

print(f"📁 Loaded data from: {data_path}")
print(f"📊 Dataset shape: {df.shape}")
print(f"🚛 Unique vehicles: {df['vehicle_id'].nunique()}")
print(f"📅 Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

In [ ]:
# Display basic info about the dataset
print("📋 Dataset Info:")
print(df.info())
print("\n" + "="*50)

print("📈 First few rows:")
df.head()

In [ ]:
# Convert timestamp to datetime and create additional features
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['fuel_efficiency'] = df['km_driven'] / df['fuel_used']
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['is_weekend'] = df['timestamp'].dt.dayofweek >= 5

print("✅ Added computed features:")
print("   - fuel_efficiency (km/L)")
print("   - date, hour, day_of_week")
print("   - is_weekend flag")

## 2. Statistical Summary

In [ ]:
# Statistical summary
print("📊 Statistical Summary:")
print("="*50)
print(df[['km_driven', 'fuel_used', 'fuel_efficiency']].describe())

In [ ]:
# Vehicle-wise summary
print("🚛 Vehicle-wise Summary:")
vehicle_summary = df.groupby('vehicle_id').agg({
    'km_driven': ['count', 'sum', 'mean'],
    'fuel_used': ['sum', 'mean'],
    'fuel_efficiency': ['mean', 'std', 'min', 'max']
}).round(2)

# Flatten column names
vehicle_summary.columns = ['_'.join(col).strip() for col in vehicle_summary.columns]
vehicle_summary = vehicle_summary.rename(columns={
    'km_driven_count': 'total_trips',
    'km_driven_sum': 'total_km',
    'km_driven_mean': 'avg_km_per_trip',
    'fuel_used_sum': 'total_fuel',
    'fuel_used_mean': 'avg_fuel_per_trip',
    'fuel_efficiency_mean': 'avg_efficiency',
    'fuel_efficiency_std': 'efficiency_std',
    'fuel_efficiency_min': 'min_efficiency',
    'fuel_efficiency_max': 'max_efficiency'
})

print(vehicle_summary)

## 3. Data Visualization

In [ ]:
# Create a comprehensive visualization dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('FleetFuel360 - Fuel Efficiency Analysis Dashboard', fontsize=16, fontweight='bold')

# 1. Fuel Efficiency Distribution
axes[0, 0].hist(df['fuel_efficiency'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].axvline(df['fuel_efficiency'].mean(), color='red', linestyle='--', 
                   label=f'Mean: {df["fuel_efficiency"].mean():.2f} km/L')
axes[0, 0].set_title('Fuel Efficiency Distribution')
axes[0, 0].set_xlabel('Fuel Efficiency (km/L)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Vehicle Performance Comparison
vehicle_efficiency = df.groupby('vehicle_id')['fuel_efficiency'].mean().sort_values(ascending=False)
axes[0, 1].bar(vehicle_efficiency.index, vehicle_efficiency.values, color='lightgreen', alpha=0.8)
axes[0, 1].set_title('Average Fuel Efficiency by Vehicle')
axes[0, 1].set_xlabel('Vehicle ID')
axes[0, 1].set_ylabel('Avg Efficiency (km/L)')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3)

# 3. Fuel Efficiency Over Time
daily_efficiency = df.groupby('date')['fuel_efficiency'].mean()
axes[1, 0].plot(daily_efficiency.index, daily_efficiency.values, marker='o', linewidth=2, markersize=4)
axes[1, 0].set_title('Daily Average Fuel Efficiency Trend')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Avg Efficiency (km/L)')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# 4. Correlation Heatmap
corr_data = df[['km_driven', 'fuel_used', 'fuel_efficiency', 'hour']].corr()
sns.heatmap(corr_data, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

## 4. Advanced Analysis

In [ ]:
# Relationship between distance and fuel consumption
plt.figure(figsize=(12, 8))

# Scatter plot with regression line for each vehicle
colors = plt.cm.Set3(np.linspace(0, 1, df['vehicle_id'].nunique()))
for i, vehicle in enumerate(df['vehicle_id'].unique()):
    vehicle_data = df[df['vehicle_id'] == vehicle]
    plt.scatter(vehicle_data['km_driven'], vehicle_data['fuel_used'], 
               alpha=0.6, label=vehicle, color=colors[i], s=50)
    
    # Add trend line
    z = np.polyfit(vehicle_data['km_driven'], vehicle_data['fuel_used'], 1)
    p = np.poly1d(z)
    plt.plot(vehicle_data['km_driven'], p(vehicle_data['km_driven']), 
             color=colors[i], linestyle='--', alpha=0.8)

plt.title('Distance vs Fuel Consumption by Vehicle', fontsize=14, fontweight='bold')
plt.xlabel('Distance Driven (km)')
plt.ylabel('Fuel Used (L)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate and display correlation
correlation = df['km_driven'].corr(df['fuel_used'])
print(f"📊 Overall Correlation (Distance vs Fuel): {correlation:.3f}")

In [ ]:
# Time-based analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Efficiency by hour of day
hourly_efficiency = df.groupby('hour')['fuel_efficiency'].mean()
axes[0].plot(hourly_efficiency.index, hourly_efficiency.values, marker='o', linewidth=2)
axes[0].set_title('Fuel Efficiency by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Avg Efficiency (km/L)')
axes[0].grid(True, alpha=0.3)

# Efficiency by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_efficiency = df.groupby('day_of_week')['fuel_efficiency'].mean().reindex(day_order)
axes[1].bar(range(len(daily_efficiency)), daily_efficiency.values, color='lightcoral', alpha=0.8)
axes[1].set_title('Fuel Efficiency by Day of Week')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Avg Efficiency (km/L)')
axes[1].set_xticks(range(len(day_order)))
axes[1].set_xticklabels([day[:3] for day in day_order], rotation=45)
axes[1].grid(True, alpha=0.3)

# Weekend vs Weekday comparison
weekend_comparison = df.groupby('is_weekend')['fuel_efficiency'].mean()
axes[2].bar(['Weekday', 'Weekend'], weekend_comparison.values, color=['steelblue', 'orange'], alpha=0.8)
axes[2].set_title('Weekday vs Weekend Efficiency')
axes[2].set_ylabel('Avg Efficiency (km/L)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Anomaly Detection Visualization

In [ ]:
# Simple anomaly detection using IQR method
def detect_outliers_iqr(series, factor=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR
    return (series < lower_bound) | (series > upper_bound)

# Detect anomalies in fuel efficiency
df['is_anomaly_iqr'] = detect_outliers_iqr(df['fuel_efficiency'])
anomalies = df[df['is_anomaly_iqr']]

print(f"🚨 Detected {len(anomalies)} anomalies ({len(anomalies)/len(df)*100:.1f}% of data)")
print(f"📊 Anomaly threshold: IQR method with factor=1.5")

# Visualize anomalies
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Anomalies over time
axes[0].scatter(df[~df['is_anomaly_iqr']]['timestamp'], 
                df[~df['is_anomaly_iqr']]['fuel_efficiency'], 
                alpha=0.6, color='blue', label='Normal', s=30)
axes[0].scatter(anomalies['timestamp'], anomalies['fuel_efficiency'], 
                alpha=0.8, color='red', label='Anomaly', s=50, marker='x')
axes[0].set_title('Fuel Efficiency Anomalies Over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Fuel Efficiency (km/L)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Anomalies by vehicle
anomaly_by_vehicle = anomalies.groupby('vehicle_id').size()
axes[1].bar(anomaly_by_vehicle.index, anomaly_by_vehicle.values, color='red', alpha=0.7)
axes[1].set_title('Number of Anomalies by Vehicle')
axes[1].set_xlabel('Vehicle ID')
axes[1].set_ylabel('Number of Anomalies')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Detailed anomaly analysis
print("🔍 Anomaly Analysis Details:")
print("="*50)

if len(anomalies) > 0:
    print(f"📊 Anomaly Statistics:")
    print(f"   - Total anomalies: {len(anomalies)}")
    print(f"   - Percentage of data: {len(anomalies)/len(df)*100:.2f}%")
    print(f"   - Efficiency range: {anomalies['fuel_efficiency'].min():.2f} - {anomalies['fuel_efficiency'].max():.2f} km/L")
    print(f"   - Mean anomaly efficiency: {anomalies['fuel_efficiency'].mean():.2f} km/L")
    print(f"   - Normal data mean: {df[~df['is_anomaly_iqr']]['fuel_efficiency'].mean():.2f} km/L")
    
    print(f"\n🚛 Vehicles with most anomalies:")
    anomaly_counts = anomalies['vehicle_id'].value_counts().head()
    for vehicle, count in anomaly_counts.items():
        total_records = len(df[df['vehicle_id'] == vehicle])
        percentage = (count / total_records) * 100
        print(f"   - {vehicle}: {count} anomalies ({percentage:.1f}% of vehicle's data)")
    
    print(f"\n📅 Most recent anomalies:")
    recent_anomalies = anomalies.nlargest(3, 'timestamp')[['vehicle_id', 'timestamp', 'fuel_efficiency', 'km_driven', 'fuel_used']]
    for _, row in recent_anomalies.iterrows():
        print(f"   - {row['vehicle_id']} on {row['timestamp'].strftime('%Y-%m-%d %H:%M')}: {row['fuel_efficiency']:.2f} km/L")
else:
    print("✅ No anomalies detected with current threshold")

## 6. Fleet Performance Insights

In [ ]:
# Generate comprehensive fleet insights
print("🚛 FLEET PERFORMANCE INSIGHTS")
print("="*50)

# Overall fleet statistics
total_km = df['km_driven'].sum()
total_fuel = df['fuel_used'].sum()
fleet_efficiency = total_km / total_fuel

print(f"📊 Fleet Overview:")
print(f"   - Total distance: {total_km:,.0f} km")
print(f"   - Total fuel consumed: {total_fuel:,.0f} L")
print(f"   - Fleet average efficiency: {fleet_efficiency:.2f} km/L")
print(f"   - Total trips: {len(df):,}")
print(f"   - Average trip distance: {df['km_driven'].mean():.1f} km")

# Best and worst performing vehicles
vehicle_performance = df.groupby('vehicle_id')['fuel_efficiency'].mean().sort_values()

print(f"\n🏆 Best Performing Vehicles:")
for vehicle, efficiency in vehicle_performance.tail(3).items():
    print(f"   - {vehicle}: {efficiency:.2f} km/L")

print(f"\n⚠️  Vehicles Needing Attention:")
for vehicle, efficiency in vehicle_performance.head(3).items():
    improvement_potential = fleet_efficiency - efficiency
    print(f"   - {vehicle}: {efficiency:.2f} km/L (improvement potential: {improvement_potential:.2f} km/L)")

# Efficiency trends
recent_data = df[df['timestamp'] >= df['timestamp'].max() - pd.Timedelta(days=3)]
older_data = df[df['timestamp'] < df['timestamp'].max() - pd.Timedelta(days=3)]

if len(recent_data) > 0 and len(older_data) > 0:
    recent_efficiency = recent_data['fuel_efficiency'].mean()
    older_efficiency = older_data['fuel_efficiency'].mean()
    efficiency_change = recent_efficiency - older_efficiency
    
    print(f"\n📈 Recent Trends:")
    if efficiency_change > 0.1:
        print(f"   ✅ Efficiency improving: +{efficiency_change:.2f} km/L")
    elif efficiency_change < -0.1:
        print(f"   ⚠️ Efficiency declining: {efficiency_change:.2f} km/L")
    else:
        print(f"   ➖ Efficiency stable: {efficiency_change:+.2f} km/L")

# Recommendations
print(f"\n💡 RECOMMENDATIONS:")
print(f"   1. Focus maintenance on vehicles with <{fleet_efficiency*0.9:.1f} km/L efficiency")
print(f"   2. Monitor vehicles with high anomaly rates for potential issues")
print(f"   3. Analyze best practices from top-performing vehicles")
if len(anomalies) > len(df) * 0.1:
    print(f"   4. High anomaly rate detected - review data quality and collection process")
print(f"   5. Consider driver training programs for efficiency improvement")

## 7. Export Results

In [ ]:
# Export analysis results
print("💾 Exporting Analysis Results...")

# Save vehicle performance summary
vehicle_summary.to_csv('../data/vehicle_performance_summary.csv')
print("   ✅ Vehicle performance summary saved")

# Save anomalies if any
if len(anomalies) > 0:
    anomalies.to_csv('../data/detected_anomalies.csv', index=False)
    print(f"   ✅ {len(anomalies)} anomalies exported")

# Save daily efficiency trends
daily_efficiency.to_csv('../data/daily_efficiency_trends.csv')
print("   ✅ Daily efficiency trends saved")

print("\n🎉 EDA Complete! Check the data/ folder for exported files.")